<h1 style="text-align: center;">WHO Life Expectancy — Modelling Experiments</h1>

This notebook focuses on experimenting with different machine learning models for predicting `Life expectancy`. The goal is to establish baseline performance, compare model behaviour, evaluate results using appropriate regression metrics, and gradually improve the modelling workflow based on evidence from the experiments.

---
## Current Feature Engineering Decision

As of **11 September 2026**, no new engineered features have been added to the dataset. The modelling experiments will initially use the original predictor variables identified during EDA.

Data cleaning and preprocessing will still be performed where required, including handling missing values, categorical variables, and any model-specific preparation.

Additional feature engineering, feature removal, transformations, or dimensionality-reduction techniques such as PCA have not yet been decided.

These decisions will be revisited after the first modelling experiments and their results are evaluated. Any later changes to the feature set will be based on model performance, validation results, and clear analytical justification rather than being introduced in advance.

---
## Data import 

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option("display.float_format", "{:.2f}".format)

In [24]:
data = pd.read_csv("../Data/Data.csv")

In [25]:
data.head(3)

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.00,263.00,62,0.01,71.28,65.00,1154,...,6.00,8.16,65.00,0.10,584.26,33736494.00,17.20,17.30,0.48,10.10
1,Afghanistan,2014,Developing,59.90,271.00,64,0.01,73.52,62.00,492,...,58.00,8.18,62.00,0.10,612.70,327582.00,17.50,17.50,0.48,10.00
2,Afghanistan,2013,Developing,59.90,268.00,66,0.01,73.22,64.00,430,...,62.00,8.13,64.00,0.10,631.74,31731688.00,17.70,17.70,0.47,9.90


In [26]:
# Remove whitespace from column names to avoid inconsistent column access
data.columns = data.columns.str.strip()

In [27]:
# Remove rows with missing target values because the target should not be imputed
data.dropna(subset=['Life expectancy'], axis=0, inplace=True)

In [28]:
data.isnull().sum()

Country                              0
Year                                 0
Status                               0
Life expectancy                      0
Adult Mortality                      0
infant deaths                        0
Alcohol                            193
percentage expenditure               0
Hepatitis B                        553
Measles                              0
BMI                                 32
under-five deaths                    0
Polio                               19
Total expenditure                  226
Diphtheria                          19
HIV/AIDS                             0
GDP                                443
Population                         644
thinness  1-19 years                32
thinness 5-9 years                  32
Income composition of resources    160
Schooling                          160
dtype: int64

In [29]:
data.isnull().mean()*100

Country                            0.00
Year                               0.00
Status                             0.00
Life expectancy                    0.00
Adult Mortality                    0.00
infant deaths                      0.00
Alcohol                            6.59
percentage expenditure             0.00
Hepatitis B                       18.89
Measles                            0.00
BMI                                1.09
under-five deaths                  0.00
Polio                              0.65
Total expenditure                  7.72
Diphtheria                         0.65
HIV/AIDS                           0.00
GDP                               15.13
Population                        21.99
thinness  1-19 years               1.09
thinness 5-9 years                 1.09
Income composition of resources    5.46
Schooling                          5.46
dtype: float64

In [30]:
x = data.drop(columns=['Life expectancy'])
y = data['Life expectancy']

In [31]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.33, random_state=1)

---

## CatBoost Baseline Model

This section establishes the first regression baseline using `CatBoostRegressor`. CatBoost is used directly with the original numerical features and the categorical features `Country` and `Status`.

No scaling or manual imputation is applied in this experiment, as CatBoost can natively handle numerical missing values and categorical features. The purpose is to measure how well the original dataset performs before introducing additional preprocessing, feature engineering, or tuning.

In [46]:
from catboost import CatBoostRegressor

In [ ]:
model = CatBoostRegressor(iterations=200, learning_rate=0.1, depth=6, loss_function='RMSE', random_seed=1, verbose=0, cat_features=["Country", "Status"])

In [48]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=6, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

### Evaluation

In [49]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 

In [50]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [51]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.9599718578378691
training score of the model is 0.9816404064208334


In [53]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 3.50445433677003
The mean absolute error is 1.2755909127036913


### Baseline Evaluation

The initial CatBoost model achieved a strong test **R² score of approximately 0.960**, compared with a training R² of approximately **0.982**. The relatively small difference suggests no obvious severe overfitting in this initial experiment.

The model produced an **MAE of approximately 1.28 years**, meaning its predictions differ from the actual Life Expectancy by about 1.28 years on average. The MSE was approximately **3.50**.

Overall, this represents a strong first baseline. However, the validation strategy and repeated country year structure should be investigated further before treating this performance as final.

---
## Country-Based Group Split

To test whether the model can generalize to completely unseen countries, the dataset will be split using `GroupShuffleSplit` with `Country` as the grouping variable.

This ensures that each country appears entirely in either the training set or the test set, preventing country overlap between both datasets.

In [54]:
from sklearn.model_selection import GroupShuffleSplit

In [55]:
spliter= GroupShuffleSplit(n_splits=1, test_size=0.33,random_state=1)

In [56]:
train_idx, test_idx = next(spliter.split(x,y,groups=x['Country']))

In [58]:
x_train = x.iloc[train_idx]
x_test = x.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [59]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=6, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

### Evaluation

In [60]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [61]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.8860535689485315
training score of the model is 0.984665543186314


In [62]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 8.804677894245565
The mean absolute error is 2.105075246356713


### Country Based Split Evaluation

The country based split achieved a test R² of approximately **0.886**, compared with about **0.960** from the original random split. This shows that the model performs worse when predicting Life Expectancy for countries that were completely unseen during training.

Despite the drop, the result is still reasonably strong and shows that the model can generalize using the remaining health, economic, and demographic features.

However, this experiment represents a much harder prediction task because `Country` is an informative feature and the model has no historical information about the test countries. Therefore, this result will be treated as an important robustness check rather than the final modelling strategy.

---

## Time Based Validation Split

This experiment will test whether the model can use historical country data to predict Life Expectancy for later years.

The training set will contain observations from **2000 to 2010**, while the test set will contain observations from **2011 to 2015**. This keeps the same countries available historically while preventing future years from appearing in the training data.

The same CatBoost model settings will be used so that any change in performance can be attributed to the time based validation strategy rather than changes in the model itself.

In [63]:
train_data = data[data["Year"] <= 2010]
test_data = data[data["Year"] > 2010]

In [66]:
x_train = train_data.drop(columns=["Life expectancy"])
y_train = train_data["Life expectancy"]

x_test = test_data.drop(columns=["Life expectancy"])
y_test = test_data["Life expectancy"]

In [67]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=6, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

### Evaluation

In [68]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [69]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.927156738476952
training score of the model is 0.9851358320353076


In [70]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 5.269819351731205
The mean absolute error is 1.5547135525841753


### Time Based Split Evaluation

The time based experiment achieved a test R² of approximately **0.927**, with a training R² of approximately **0.985**. The model also produced an MAE of approximately **1.55 years** and an MSE of approximately **5.27**.

This performs better than the unseen country experiment, although it is slightly weaker than the original random split. More importantly, this validation setup is more realistic because the model learns from historical observations between 2000 and 2010 and is then tested on future observations from 2011 to 2015.

The results show that CatBoost is able to learn useful historical patterns and generalize them reasonably well to later years.

For this project, the time based validation strategy appears to be the most suitable direction for the final model because it better reflects the intended task of predicting future Life Expectancy from previously observed country data.

---

## CatBoost Hyperparameter Tuning

This section experiments with multiple CatBoost parameter combinations while keeping the same time based validation strategy.

Different settings will be compared using the same evaluation metrics, and only the parameter combination that produces the best validation performance will be kept as the tuned model.

In [121]:
model = CatBoostRegressor(iterations=200, learning_rate=0.1, depth=5, loss_function='RMSE', random_seed=1, verbose=0, cat_features=["Country", "Status"])

In [122]:
model.fit(x_train, y_train)

CatBoostRegressor(cat_features=['Country', 'Status'], depth=5, iterations=200, learning_rate=0.1, loss_function='RMSE', random_seed=1, verbose=0)

In [123]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [124]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.9287859421264668
training score of the model is 0.9806376910274035


In [125]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 5.151955204236823
The mean absolute error is 1.5368232732825964


---

## XGBoost Regression Experiment

This section focuses only on experimenting with `XGBRegressor` as a comparison model against CatBoost.

The same time based training and test split will be kept so that the comparison remains fair. XGBoost will be trained and evaluated using the same regression metrics, allowing its performance to be compared directly with the tuned CatBoost model.

In [126]:
from xgboost import XGBRegressor

In [171]:
model =  XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, min_child_weight=5, gamma=0.1, reg_alpha=5, enable_categorical=True, random_state=1)

In [129]:
cat_cols = ["Country", "Status"]

for col in cat_cols:
    x_train[col] = x_train[col].astype("category")
    x_test[col] = pd.Categorical(
        x_test[col],
        categories=x_train[col].cat.categories
    )

In [172]:
model.fit(x_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [173]:
y_hat = model.predict(x_test)
y_hat_train = model.predict(x_train)

In [174]:
print(f"Accuracy of the model is {r2_score(y_test, y_hat)}")
print(f"training score of the model is {r2_score(y_train, y_hat_train)}")

Accuracy of the model is 0.9179372887113636
training score of the model is 0.9948236389595554


In [175]:
print(f'Mean squared error is {mean_squared_error(y_test, y_hat)}\nThe mean absolute error is {mean_absolute_error(y_test, y_hat)}')

Mean squared error is 5.936797103292195
The mean absolute error is 1.5215454118238771


### XGBoost Evaluation

The tuned XGBoost model achieved a test R² of approximately **0.918**, with a training R² of approximately **0.995**. It produced an MSE of approximately **5.94** and an MAE of approximately **1.52 years**.

The model therefore performs well on the future test period. However, its test R² and MSE are weaker than those achieved by the tuned CatBoost model. The much higher training score also indicates that XGBoost is fitting the training data more strongly and generalizing slightly less effectively.

Although XGBoost achieved a slightly lower MAE, CatBoost provides the stronger overall balance between predictive performance and generalization. Therefore, CatBoost will remain the primary model for the next stage of the project.